<a href="https://colab.research.google.com/github/yufree/OSSpRe/blob/main/OSSpRe_colab_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OSSpRe Verification Notebook (Colab / Linux, renv environment restore)

This notebook verifies the OSSpRe fixes on Linux. The R environment is restored
strictly from the repository's `renv.lock` (R 4.6.1, 178 packages);
`opentimsr` is restored from the author's drat repository (vendored build,
which works around the symlink problem in the upstream GitHub monorepo).

**No rpy2 dependency** — all R logic runs through `Rscript`, so there is no
embedded-R initialization to crash.

Workflow:

1. Install system libraries + **R 4.6.1** from Posit r-builds (matches `renv.lock`)
2. Clone the repository from GitHub (fixes already merged into `main`),
   verify markers, and fall back to the drat lockfile entry if needed
3. `renv::restore()` all R packages (CRAN via P3M binaries; the 42
   Bioconductor 3.23 packages compile from source — **the slowest step,
   about 30-50 minutes**)
4. Run the test suite: compile the 4 C++ files → `findpeakalign` correctness
   and performance → `find_2d_peaks` planted-peak recovery + OpenMP
   determinism → Bruker library 1/k0 <-> CCS round trip → `helper.r`
   end-to-end
5. Produce a PASS/FAIL/SKIP summary report

**Usage**: open directly in Colab:
`https://colab.research.google.com/github/yufree/OSSpRe/blob/main/OSSpRe_colab_test.ipynb`
A default CPU runtime is enough. The full run takes **about 40-60 minutes**;
keep the tab open to avoid disconnects.

In [ ]:
%%bash
# ---- system libraries + R 4.6.1 from Posit r-builds (matches renv.lock), ~3 min ----
set -e
apt-get update -qq
PKGS="build-essential gfortran git curl ca-certificates \
  libcurl4-openssl-dev libssl-dev libxml2-dev libzstd-dev \
  libfontconfig1-dev libfreetype6-dev pkg-config libpng-dev libtiff-dev \
  libjpeg-dev libnetcdf-dev libgit2-dev libglpk-dev libharfbuzz-dev \
  libfribidi-dev libcairo2-dev libxt-dev libblas-dev liblapack-dev"
DEBIAN_FRONTEND=noninteractive apt-get install -y --no-install-recommends $PKGS \
  > /tmp/apt.log 2>&1 || { echo "some system packages failed to install, continuing..."; tail -5 /tmp/apt.log; }

. /etc/os-release
case "$VERSION_CODENAME" in
  noble) UB=2404 ;;
  *)     UB=2204 ;;
esac
echo "$VERSION_CODENAME" > /content/codename
echo "Ubuntu $VERSION_CODENAME -> ubuntu-$UB"
if curl -fsSL -o /tmp/r.deb "https://cdn.posit.co/r/ubuntu-${UB}/pkgs/r-4.6.1_1_amd64.deb" \
   && DEBIAN_FRONTEND=noninteractive apt-get install -y /tmp/r.deb >> /tmp/apt.log 2>&1; then
  ln -sf /opt/R/4.6.1/bin/R /usr/local/bin/R
  ln -sf /opt/R/4.6.1/bin/Rscript /usr/local/bin/Rscript
  echo "installed r-builds R 4.6.1 (matches renv.lock)"
else
  echo "warning: r-builds install failed, falling back to apt r-base (version may not match the lockfile)"
  DEBIAN_FRONTEND=noninteractive apt-get install -y r-base r-base-dev >> /tmp/apt.log 2>&1
fi
R --version | head -1
nproc

## 1. Clone the repository and verify the fixes are included

The fixes are merged into GitHub `main` (code fixes >= `79cb81b`, drat
environment fix >= `db86bf8`). After cloning we verify the markers; if
`renv.lock` is still the old version (should not happen), the opentimsr entry
is switched to drat automatically.

In [ ]:
%%bash
set -e
cd /content
rm -rf OSSpRe
git clone --depth 1 https://github.com/yufree/OSSpRe.git
cd OSSpRe
git log --oneline -1
grep -q "roi_idx" helper.r \
  && echo "code fixes included (>= 79cb81b)" \
  || { echo "error: the cloned repository is missing the fixes"; exit 1; }
grep -q 'yufree.github.io/drat' renv.lock && echo "renv.lock already points to drat" || python3 - <<'PYEOF'
import json
lock = json.load(open('renv.lock'))
p = lock['Packages'].get('opentimsr')
if p:
    lock['Packages']['opentimsr'] = {k: v for k, v in p.items()
                                     if not k.startswith(('Remote', 'Github'))}
    lock['Packages']['opentimsr']['Source'] = 'Repository'
    repos = lock['R']['Repositories']
    if not any(r['Name'] == 'drat' for r in repos):
        repos.append({'Name': 'drat', 'URL': 'https://yufree.github.io/drat'})
    json.dump(lock, open('renv.lock', 'w'), indent=2, ensure_ascii=False)
    print('renv.lock: opentimsr entry switched to drat')
PYEOF
echo "clone ready"

## 2. `renv::restore()` the R environment from the lockfile (about 30-50 minutes)

- 178 packages: the 134 CRAN packages come as Posit P3M **binaries** (minutes);
  the 42 Bioconductor 3.23 packages have no binaries and **compile from
  source** (mzR/MSnbase/xcms are the slow ones)
- `pmd` / `rmwf` are restored from **GitHub** tarballs per the lockfile
  Remote fields; `opentimsr` is restored from the **drat repository**
  (https://yufree.github.io/drat, vendored build)
- If a pinned CRAN version has been updated meanwhile, renv falls back to the
  cloud.r-project.org archive automatically
- If the run is interrupted (disconnect / network hiccup), **just re-run this
  cell**: renv only installs the missing packages

In [ ]:
%%writefile /content/setup_env.R
# Restore the environment from renv.lock (standalone R process, no rpy2 needed)
# Configure renv before loading it to force source builds for packages whose
# P3M binaries are broken on the current Colab image
Sys.setenv(RENV_CONFIG_TYPE = "source")
options(renv.config.type = "source")

codename <- tryCatch(readLines("/content/codename", warn = FALSE), error = function(e) "jammy")
repos <- c(P3Mbin  = sprintf("https://packagemanager.posit.co/all/__linux__/%s/latest", codename),
           drat    = "https://yufree.github.io/drat",
           CRAN    = "https://cloud.r-project.org",
           BioCsoft = "https://bioconductor.org/packages/3.23/bioc")
options(repos = repos)
if (!requireNamespace("renv", quietly = TRUE)) install.packages("renv", repos = repos[1])

setwd("/content/OSSpRe")
source("renv/activate.R")
cat("R:", as.character(getRversion()), "| project library:", unlist(.libPaths())[1], "\n")

t0 <- Sys.time()
# Rebuild stringi explicitly from source to bypass any broken binary cache
cat("compiling stringi from source...\n")
try(renv::install("stringi", rebuild = TRUE, prompt = FALSE))

# Regular restore
renv::restore(prompt = FALSE, repos = repos)
cat(sprintf("renv::restore finished in %.1f minutes\n",
            as.numeric(difftime(Sys.time(), t0, units = "mins"))))

need <- c("Rcpp", "data.table", "opentimsr", "pmd", "rmwf", "enviGCMS", "irlba", "ClusterR")
got <- sapply(need, requireNamespace, quietly = TRUE)
print(got)
stopifnot(all(got))
cat("environment ready\n")

In [ ]:
%%bash
# ~30-50 minutes (Bioconductor source compilation); if interrupted, just re-run this cell
# Install system-level build dependencies required by R packages (e.g. libglpk-dev)
sudo apt-get update -qq && sudo apt-get install -y --no-install-recommends libglpk-dev libgmp3-dev libuv1-dev
export RENV_CONFIG_INSTALL_TRANSACTIONAL=FALSE
Rscript /content/setup_env.R

## 3. Run the full test suite (about 2-5 minutes)

Coverage: C++ compilation (Linux/gcc/OpenMP), `findpeakalign` line-by-line
comparison against a brute-force reference implementation + edge cases +
performance benchmarks, `find_2d_peaks` planted-peak recovery + OpenMP
concurrency determinism, Bruker library 1/k0 <-> CCS round trip (auto-SKIP if
the download is unavailable), and `helper.r` end-to-end (PCA segmentation
dimension, peak counting, ROI clustering NaN guards, annotation, empty-network
guard, function signatures).

In [ ]:
%%writefile /content/run_tests.R
# OSSpRe fix verification (standalone R process; state is passed via files)
setwd("/content/OSSpRe")
source("renv/activate.R")   # use the restored project library
pdf(NULL)                   # silence interactive plot devices; explicit png() is unaffected

RESULTS <- new.env()
check <- function(name, expr) {
  ok <- tryCatch(isTRUE(force(expr)),
                 error = function(e) {
                   cat("  ERROR:", conditionMessage(e), "\n")
                   FALSE
                 })
  RESULTS[[name]] <- ok
  cat(sprintf("[%s] %s\n", if (ok) "PASS" else "FAIL", name))
  invisible(ok)
}

## ---- S1: compile the 4 C++ files ----
check("cpp_one_over_k0_compiles", { library(Rcpp); sourceCpp("one_over_k0_to_ccs.cpp"); TRUE })
check("cpp_ccs_to_k0_compiles",   { sourceCpp("ccs_to_one_over_k0.cpp"); TRUE })
check("cpp_peakalign_compiles",   { sourceCpp("peakalign.cpp"); TRUE })
check("cpp_peakfinder_compiles",  { sourceCpp("peak_finder.cpp"); TRUE })

# R wrapper from helper.r (defined early for S3; helper.r redefines it later, harmless)
find_2d_peaks <- function(mz, ccs, intensity, mz_ppm = 20, ccs_tolerance = 0.03,
                          snr = 3.0, mz_bins = 1000, ccs_bins = 50) {
  if (!is.numeric(mz) || !is.numeric(ccs) || !is.numeric(intensity))
    stop("All inputs must be numeric vectors")
  if (length(mz) != length(ccs) || length(mz) != length(intensity))
    stop("All input vectors must have the same length")
  find_2d_peaks_spatial_parallel_openmp(mz, ccs, intensity,
                                        mz_ppm, ccs_tolerance, snr, mz_bins, ccs_bins)
}

## ---- S2: findpeakalign correctness + performance ----
set.seed(123)
n <- 4000
frame <- sample(1:3, n, replace = TRUE)
mz <- round(runif(n, 700, 800), 4)
ccs <- round(runif(n, 250, 320), 2)
intensity <- round(runif(n) * 1000, 1)
ref_mz <- c(710.05, 725.5, 733.33, 748.9, 760.02, 775.6, 790.123, 799.999)
ref_ccs <- c(280.1, 290.5, 300.2, 265.0, 310.3, 275.8, 295.4, 305.9)
ppm <- 20; ccs_shift <- 0.03

ref_impl <- function() {
  out <- data.frame(frame = integer(), mz_ccs = character(), intensity = double())
  for (i in seq_along(ref_mz)) {
    mz_tol <- ppm * ref_mz[i] / 1e6; ccs_tol <- ccs_shift * ref_ccs[i]
    key <- paste0(sprintf("%.6f", ref_mz[i]), "_", sprintf("%.6f", ref_ccs[i]))
    for (f in sort(unique(frame))) {
      j <- which(frame == f)
      hit <- abs(mz[j] - ref_mz[i]) <= mz_tol & abs(ccs[j] - ref_ccs[i]) <= ccs_tol
      tot <- sum(intensity[j][hit])
      if (tot > 0) out <- rbind(out, data.frame(frame = f, mz_ccs = key, intensity = tot))
    }
  }
  out
}
r_new <- as.data.frame(findpeakalign(mz, ccs, intensity, frame, ref_mz, ref_ccs, ppm, ccs_shift))
r_ref <- ref_impl()
a <- r_new[order(r_new$frame, r_new$mz_ccs), ]; b <- r_ref[order(r_ref$frame, r_ref$mz_ccs), ]
check("align_matches_reference", nrow(a) == nrow(b) && identical(a$frame, b$frame) &&
        identical(a$mz_ccs, b$mz_ccs) && max(abs(a$intensity - b$intensity)) == 0)

e <- findpeakalign(numeric(0), numeric(0), numeric(0), integer(0), ref_mz, ref_ccs, ppm, ccs_shift)
z <- findpeakalign(mz, ccs, intensity, frame, 999.9, 200, ppm, ccs_shift)
check("align_edgecases", nrow(e) == 0 && nrow(z) == 0)

for (nn in c(2e4, 1e5, 5e5)) {
  mz2 <- round(runif(nn, 700, 800), 4); ccs2 <- round(runif(nn, 250, 320), 2)
  int2 <- runif(nn); fr2 <- sample(1:500, nn, replace = TRUE)
  rmz <- round(runif(500, 700, 800), 4); rccs <- runif(500, 250, 320)
  t0 <- Sys.time()
  res <- findpeakalign(mz2, ccs2, int2, fr2, rmz, rccs, ppm, ccs_shift)
  cat(sprintf("  %d points x 500 ref peaks x 500 frames: %.3f s (%d rows matched)\n",
              nn, as.numeric(difftime(Sys.time(), t0, units = "secs")), nrow(res)))
}
t0 <- Sys.time(); invisible(ref_impl())
cat(sprintf("  [reference] the brute-force R implementation already takes %.1f s for just %d points x 8 peaks x 3 frames\n",
            as.numeric(difftime(Sys.time(), t0, units = "secs")), n))
check("align_benchmark_runs", TRUE)

## ---- S3: find_2d_peaks planted recovery + concurrency determinism ----
# The background mimics the density of real super-pixel data: point spacing is
# far below the m/z (20 ppm ~ 0.015 Da) and CCS (3% ~ 8.5) windows; otherwise
# isolated points would bypass the SNR / local-maximum checks
set.seed(1)
n_bg <- 20000
mzp <- runif(n_bg, 748, 752); ccsp <- runif(n_bg, 280, 300); intp <- runif(n_bg, 1, 100)
# 5 planted peaks: 0.8 Da apart in m/z (far beyond the 2x window), intensities
# 5000-10000 >> background
centers <- data.frame(mz = seq(748.4, 751.6, length.out = 5), ccs = 290)
for (i in 1:5) {
  mzp <- c(mzp, centers$mz[i] + rnorm(30, 0, centers$mz[i] * 5e-6))
  ccsp <- c(ccsp, centers$ccs[i] + rnorm(30, 0, centers$ccs[i] * 0.005))
  intp <- c(intp, c(10000, runif(29, 5000, 9999)))
}
res2 <- find_2d_peaks(mzp, ccsp, intp, mz_ppm = 20, ccs_tolerance = 0.03, snr = 3,
                      mz_bins = 1000, ccs_bins = 50)
found <- sapply(1:5, function(i)
  any(abs(res2$mz - centers$mz[i]) <= centers$mz[i] * 50e-6 &
      abs(res2$ccs - centers$ccs[i]) <= centers$ccs[i] * 0.05))
cat("  peaks found:", nrow(res2), "(background should be removed by SNR); all 5 planted peaks recovered:", all(found), "\n")
check("peakfinder_recovers_planted", nrow(res2) <= 10 && all(found))

r5 <- replicate(5, find_2d_peaks(mzp, ccsp, intp, 20, 0.03, 3, 1000, 50), simplify = FALSE)
check("peakfinder_deterministic_openmp", all(sapply(r5[-1], function(x) identical(x, r5[[1]]))))

d <- find_2d_peaks(rep(700.5, 10), rep(280, 10), 1:10, 20, 0.03, 3, 100, 50)
check("peakfinder_degenerate_input", nrow(d) >= 1 && !any(is.na(d$ccs)))

## ---- S4: Bruker library round trip (optional, auto-SKIP if download fails) ----
lib <- tryCatch(opentimsr::download_bruker_proprietary_code("/content/bruker"),
               error = function(e) NULL)
if (is.null(lib) || !file.exists("/content/bruker/libtimsdata.so")) {
  RESULTS[["bruker_roundtrip"]] <- NA
  cat("[SKIP] bruker_roundtrip (Bruker library download unavailable)\n")
} else {
  cat("Bruker library:", lib, "\n")
  check("bruker_roundtrip", {
    k0 <- seq(0.6, 1.6, length.out = 50); mzv <- seq(200, 1000, length.out = 50)
    ccs3 <- one_over_k0_to_ccs_parallel(k0, rep(1, 50), mzv, lib)
    k0b <- ccs_to_one_over_k0_parallel(ccs3, rep(1, 50), mzv, lib)
    max(abs(k0 - k0b)) < 1e-4 &&
      is.finite(sum(one_over_k0_to_ccs_parallel(k0, rep(1, 50), mzv, lib)))
  })
}

## ---- S5: helper.r end-to-end ----
t0 <- Sys.time()
suppressPackageStartupMessages(source("helper.r"))
cat(sprintf("helper.r loaded (%.1fs)\n", as.numeric(difftime(Sys.time(), t0, units = "secs"))))

set.seed(42)
n_pix <- 200; n_ion <- 60
xs <- rep(1:20, each = 10) * 10; ys <- rep(1:10, 20) * 10
mzv <- round(seq(700, 900, length.out = n_ion), 4)
imv <- round(seq(250, 320, length.out = n_ion), 2)
M <- sapply(seq_len(n_ion), function(i) {
  if (i <= 20) as.numeric(xs + rnorm(n_pix, 0, .3))
  else if (i <= 40) as.numeric(ys + rnorm(n_pix, 0, .3))
  else rep(0, n_pix)
})
dt <- data.table(location = paste0(xs, "_", ys), M)
colnames(dt) <- c("location", paste0(mzv, "_", imv))
csv <- "test_ticmzccs.csv"; fwrite(dt, csv)

s <- capture.output(getsummary(csv))
check("helper_getsummary", any(grepl("m/z range", s)))

invisible(capture.output(plot_peak_stats(csv)))
d2 <- fread(csv)
np_fn <- d2[, rowSums(.SD != 0), .SDcols = -1]
np_manual <- rowSums(as.matrix(d2[, -1]) != 0)
check("helper_np_counts_correct", isTRUE(all.equal(np_fn, np_manual)))

invisible(capture.output(perform_pca_segmentation(csv, "test_seg.csv")))
seg <- fread("test_seg.csv")
check("helper_pca_clusters_pixels", nrow(seg) == n_pix && length(unique(seg$seg)) >= 2)

perform_reactomics_analysis(csv, "test_react.csv")
rr <- fread("test_react.csv")
check("helper_reactomics_structure", nrow(rr) == n_pix && ncol(rr) == 2 + 3 * 4)

locindex <- rep(c(TRUE, FALSE), each = n_pix / 2)
okc <- tryCatch({
  suppressWarnings(perform_cluster_ions(csv, "test_ioncluster.csv", locindex = locindex,
                                        hclust_cutoff = 0.6, min_cluster_size = 3,
                                        folder_name = "test_cluster"))
  TRUE
}, error = function(e) { cat("  error:", conditionMessage(e), "\n"); FALSE })
ic <- if (file.exists("test_ioncluster.csv")) fread("test_ioncluster.csv") else data.table()
check("helper_cluster_roi_no_nan_crash", okc && nrow(ic) == n_ion)

sel <- fread("lipidall.csv")[adducts %in% c("[M+H]", "[M+Na]")][1:5]
dt6 <- data.table(location = paste0(xs, "_", ys), matrix(runif(n_pix * 5), nrow = n_pix))
colnames(dt6) <- c("location", paste0(sel$mz, "_", sel$ccs))
fwrite(dt6, "test_peak5.csv")
getanno(database = "lipidall.csv", mode = "pos", peakpath = "test_peak5.csv", annofile = "test_anno.csv")
check("helper_getanno", nrow(fread("test_anno.csv")) >= 5)

fwrite(data.frame(name = character(), mz = numeric()), "test_emptyanno.csv")
okn <- tryCatch({
  generate_molecular_network(csv, "test_ioncluster.csv", "test_emptyanno.csv", "test_net.csv"); TRUE
}, error = function(e) FALSE)
check("helper_network_empty_guard", okn && nrow(fread("test_net.csv")) == 0)

fg <- formals(getqlist); fr <- formals(getrefpeak)
check("helper_signatures", all(c("mz_ppm", "ccs_tolerance") %in% names(fg)) &&
        identical(fr$libpath, "./") && !exists("perform_roi_reactomics_analysis"))

## ---- summary ----
res_list <- as.list(RESULTS)
sm <- data.frame(test = names(res_list), pass = unlist(res_list, use.names = FALSE))
print(sm, row.names = FALSE)
cat(sprintf("\n%d PASS / %d FAIL / %d SKIP\n",
            sum(sm$pass, na.rm = TRUE), sum(!sm$pass, na.rm = TRUE), sum(is.na(sm$pass))))
write.csv(sm, "/content/test_summary.csv", row.names = FALSE)

In [ ]:
%%bash
cd /content/OSSpRe
Rscript /content/run_tests.R

## 4. Summary report

In [ ]:
import pandas as pd
from IPython.display import Image, display
import os

s = pd.read_csv('/content/test_summary.csv')
def color(v):
    if pd.isna(v): return 'color: gray'
    return 'color: green' if v else 'color: red; font-weight: bold'
if hasattr(s.style, 'map'):
    display(s.style.map(color, subset=['pass']))
else:
    display(s.style.applymap(color, subset=['pass']))

# show one cluster image generated by the ROI ion clustering (visual check)
for p in ['/content/OSSpRe/test_cluster']:
    if os.path.isdir(p):
        pngs = sorted(f for f in os.listdir(p) if f.endswith('.png'))
        if pngs:
            print('sample cluster image:', pngs[0])
            display(Image(os.path.join(p, pngs[0]), width=300))

## Interpreting the results

- **All PASS**: every fix is verified on Linux + gcc/OpenMP + the renv-restored
  author environment (+ the real Bruker library when downloadable).
- **bruker_roundtrip SKIP**: the Bruker download is unavailable from Colab;
  this does not affect any other conclusion.
- Any **FAIL**: look for the `ERROR:` lines in the `run_tests.R` cell output
  above and report them together with `/content/test_summary.csv`.
- Interrupted environment restore: re-run the `Rscript` cell in section 2;
  renv resumes and only installs missing packages.

## Testing with real data (optional)

The C++ and algorithm layers are fully covered above; `getrefpeak`/`getqlist`
require real Bruker `.d` data:

```r
# upload a raw.d zip to /content first, then in R:
system("cd /content && unzip -q raw.zip")
path <- "/content/raw.d"
getrefpeak(path = path, accept_Bruker_EULA_and_on_Windows_or_Linux = TRUE,
           libpath = "/content/bruker/", batch_size = 100000,
           outref = "refpeak.csv", outcoord = "coord.csv",
           xrange = c(0, 1e9), yrange = c(0, 1e9))
# ...then follow the workflow.Rmd steps
```

Note that `workflow.Rmd` defaults to `eval = FALSE` (tutorial mode); run the
chunks one by one or switch the setup chunk to `eval = TRUE`.